# Step 0 — Crop faces from raw/ (ported from training/cropping.py)

Runs MTCNN face detection on every image in `../data/raw/<class>/` and saves ONLY the detected face region into `data/cropped/<class>/` — same logic as the standalone `cropping.py` script, now folded into the notebook so re-running from here always reflects the CURRENT contents of `data/raw/`, not a stale snapshot from whenever `cropping.py` was last run separately.

**Always replaces, never skips**: each class folder is cleared before writing, every run, unconditionally — no need to manually delete anything first, even if you've added/removed images in `data/raw/` since the last run.

Same rules as before: keeps only the LARGEST detected face if multiple are found, crops to the exact MTCNN box (no margin, matches what the live app sends at inference), and skips (logs) any image with no detectable face.

In [7]:
import os
import sys
import glob
import numpy as np
from PIL import Image

sys.path.insert(0, '..')  # so src.face_detector is importable
from src.face_detector import detect_faces

RAW_DIR = '../data/raw'
OUT_DIR = '../data/cropped'

def load_rgb(path):
    img = Image.open(path)
    if img.mode != 'RGB':
        img = img.convert('RGB')
    return img, np.array(img)

def pick_largest_face(boxes):
    return max(boxes, key=lambda b: b[2] * b[3])

def clear_dir(path):
    if os.path.isdir(path):
        for f in glob.glob(os.path.join(path, '*')):
            os.remove(f)

class_dirs = sorted(d for d in os.listdir(RAW_DIR) if os.path.isdir(os.path.join(RAW_DIR, d)))
print(f"Found {len(class_dirs)} classes in {RAW_DIR}/\n")

no_face_log = []
total_seen = 0
total_saved = 0

for class_name in class_dirs:
    src_files = sorted(glob.glob(os.path.join(RAW_DIR, class_name, '*')))
    out_dir = os.path.join(OUT_DIR, class_name)
    os.makedirs(out_dir, exist_ok=True)
    clear_dir(out_dir)  # always replace - reflects current raw/, not a stale prior run

    saved_this_class = 0
    for f in src_files:
        total_seen += 1
        try:
            img, arr = load_rgb(f)
        except Exception as e:
            print(f"  could not open {f}: {e}")
            continue

        boxes = detect_faces(arr)
        if len(boxes) == 0:
            no_face_log.append(f)
            continue

        x, y, w, h = pick_largest_face(boxes)
        x, y = max(0, x), max(0, y)
        face_crop = img.crop((x, y, x + w, y + h))

        out_path = os.path.join(out_dir, f"{saved_this_class:03d}.png")
        face_crop.save(out_path)
        saved_this_class += 1
        total_saved += 1

    print(f"  {class_name:20s} {saved_this_class}/{len(src_files)} image(s) had a detectable face")

print(f"\nDone. {total_saved}/{total_seen} images saved to {OUT_DIR}/")
if no_face_log:
    print(f"\n{len(no_face_log)} image(s) had NO detectable face (skipped):")
    for f in no_face_log:
        print(f"  {f}")

Found 15 classes in ../data/raw/

  Arya_Stark           36/36 image(s) had a detectable face
  Brandon_Stark        26/26 image(s) had a detectable face
  Catelyn_Stark        29/29 image(s) had a detectable face
  Cersei_Lannister     20/20 image(s) had a detectable face
  Eddard_Stark         43/43 image(s) had a detectable face
  Jaime_Lannister      25/25 image(s) had a detectable face
  Joffrey_Baratheon    27/27 image(s) had a detectable face
  Jon_Snow             29/30 image(s) had a detectable face
  Maester_Luwin        24/24 image(s) had a detectable face
  Rickon_Stark         20/20 image(s) had a detectable face
  Robb_Stark           21/23 image(s) had a detectable face
  Robert_Baratheon     27/27 image(s) had a detectable face
  Rodrik_Cassel        24/24 image(s) had a detectable face
  Sansa_Stark          25/25 image(s) had a detectable face
  Theon_Greyjoy        25/25 image(s) had a detectable face

Done. 401/404 images saved to data/cropped/

3 image(s) had NO de

# Step 1 — Split into train / val / test (60/20/20 by percentage, per class, shuffled)

Takes `data/cropped/` directly — **every real image, no cap** (earlier this read from a 20-per-class-capped `data/cropped_20/`, which was throwing away 108 of 408 real images, 26% of the dataset, for no real benefit). Class imbalance gets handled later at training time via `class_weight`, not by deleting data here.

Each class is split by **percentage** (60% train / 20% val / 20% test) of however many real images it actually has — not a fixed count. So `Eddard_Stark` (43 real) and `Cersei_Lannister` (20 real) both get a fair 60/20/20 split of their own real data, instead of both being forced to the same flat number.

Each class's images are shuffled (fixed seed, reproducible) before slicing — not taken in original order, since consecutive images can be near-identical consecutive video frames.

All three folders (`data/splits/train/`, `data/splits/val/`, `data/splits/test/`) are written fresh each run. `data/cropped/` is only read, never modified.

In [8]:
import os
import glob
import random
import shutil

random.seed(42)  # reproducible split

SRC_DIR = '../data/cropped'
BASE_DIR = '../data/splits'
TRAIN_DIR = os.path.join(BASE_DIR, 'train')
VAL_DIR = os.path.join(BASE_DIR, 'val')
TEST_DIR = os.path.join(BASE_DIR, 'test')
TRAIN_FRAC = 0.60
VAL_FRAC = 0.20
# TEST_FRAC is whatever's left over, computed as a remainder below - this
# guarantees train+val+test always sums to exactly the class's real total.

def clear_dir(path):
    if os.path.isdir(path):
        for f in glob.glob(os.path.join(path, '*')):
            os.remove(f)

class_dirs = sorted(d for d in os.listdir(SRC_DIR) if os.path.isdir(os.path.join(SRC_DIR, d)))
print(f"Found {len(class_dirs)} classes in {SRC_DIR}/\n")

for class_name in class_dirs:
    src_files = sorted(glob.glob(os.path.join(SRC_DIR, class_name, '*')))
    random.shuffle(src_files)  # shuffle BEFORE splitting - avoids frame-order/scene bias

    n_total = len(src_files)
    n_train = max(1, round(n_total * TRAIN_FRAC))
    n_val = max(1, round(n_total * VAL_FRAC))
    n_test = n_total - n_train - n_val
    if n_test < 1:
        # safety net for very small classes - borrow from train so val/test never end up empty
        deficit = 1 - n_test
        n_train -= deficit
        n_test = 1

    train_files = src_files[:n_train]
    val_files = src_files[n_train:n_train + n_val]
    test_files = src_files[n_train + n_val:n_train + n_val + n_test]

    for out_dir_name, files in [(TRAIN_DIR, train_files), (VAL_DIR, val_files), (TEST_DIR, test_files)]:
        out_dir = os.path.join(out_dir_name, class_name)
        os.makedirs(out_dir, exist_ok=True)
        clear_dir(out_dir)
        for i, f in enumerate(files):
            ext = os.path.splitext(f)[1]
            shutil.copy(f, os.path.join(out_dir, f"{i:03d}{ext}"))

    print(f"  {class_name:20s} {n_total} total -> {len(train_files)} train / {len(val_files)} val / {len(test_files)} test")

print(f"\nDone. {TRAIN_DIR}/, {VAL_DIR}/, {TEST_DIR}/ created.")

Found 15 classes in data/cropped/

  Arya_Stark           36 total -> 22 train / 7 val / 7 test
  Brandon_Stark        26 total -> 16 train / 5 val / 5 test
  Catelyn_Stark        29 total -> 17 train / 6 val / 6 test
  Cersei_Lannister     20 total -> 12 train / 4 val / 4 test
  Eddard_Stark         43 total -> 26 train / 9 val / 8 test
  Jaime_Lannister      25 total -> 15 train / 5 val / 5 test
  Joffrey_Baratheon    27 total -> 16 train / 5 val / 6 test
  Jon_Snow             29 total -> 17 train / 6 val / 6 test
  Maester_Luwin        24 total -> 14 train / 5 val / 5 test
  Rickon_Stark         20 total -> 12 train / 4 val / 4 test
  Robb_Stark           21 total -> 13 train / 4 val / 4 test
  Robert_Baratheon     27 total -> 16 train / 5 val / 6 test
  Rodrik_Cassel        24 total -> 14 train / 5 val / 5 test
  Sansa_Stark          25 total -> 15 train / 5 val / 5 test
  Theon_Greyjoy        25 total -> 15 train / 5 val / 5 test

Done. data/splits\train/, data/splits\val/, data/

# Step 2 — Augment TRAIN ONLY (10x: 1 original + 9 augmented); val/test copied untouched

Takes `data/splits/train/`, `data/splits/val/`, `data/splits/test/` (from Step 1) and writes `data/augmented/train/`, `data/augmented/val/`, `data/augmented/test/`.

- **`data/splits/train/` gets augmented 10x** (1 original + 9 variants) — this is the only split that ever gets touched by augmentation.
- **`data/splits/val/` and `data/splits/test/` are copied as-is** — real images only, no augmentation, no multiplication. A class with 8 real test images stays 8 test images. This keeps val/test as an honest, untouched signal of real-world performance.

## The 9 properties (train only, combined per copy, not isolated)

Each of the 9 output copies runs through ALL 9 properties below, and EACH property independently rolls its own 50% coin-flip to decide if it applies to that specific copy.

| # | Property | Range | Why (for facial data) |
|---|---|---|---|
| 1 | Horizontal flip | mirror | faces are roughly symmetric |
| 2 | Rotation | +/-15 deg | head-tilt |
| 3 | Shear | +/-10 deg | viewing-angle perspective |
| 4 | Zoom/crop | 90-110% (both zoom in and out) | subject distance from camera |
| 5 | Translation | +/-10% | imperfect crop centering |
| 6 | Brightness | 0.7-1.3x | GoT is dark/moody-lit |
| 7 | Contrast | 0.7-1.3x | different camera/scene settings |
| 8 | Gaussian blur | radius 0.5-1.2 | source is compressed video, some crops already low-res |
| 9 | Saturation | 0.85-1.15x | camera color-calibration variation |

`data/splits/train/`, `data/splits/val/`, `data/splits/test/` are only read here, never modified.

In [9]:
import os
import glob
import math
import random
import time
from PIL import Image, ImageEnhance, ImageFilter

random.seed(42)  # reproducible augmentation

SRC_SPLITS = ['train', 'val', 'test']
SRC_BASE_DIR = '../data/splits'
OUT_DIR = '../data/augmented'
N_AUG_PER_IMAGE = 9  # + the original = 10x total per real image, TRAIN ONLY
THRESHOLD = 0.5  # every property has an independent 50% chance of applying

def clear_dir(path):
    # Every run writes the SAME deterministic filenames (fixed seed), so
    # save() overwrites existing files directly - deleting first is only a
    # nice-to-have cleanup, never required for correctness. If OneDrive or
    # antivirus holds a file locked longer than this retry budget, skip it
    # (warn) instead of crashing - it gets overwritten by the save loop anyway.
    if os.path.isdir(path):
        for f in glob.glob(os.path.join(path, '*')):
            for attempt in range(5):
                try:
                    os.remove(f)
                    break
                except PermissionError:
                    if attempt == 4:
                        print(f"    (skipping delete of locked file, will be overwritten: {f})")
                    else:
                        time.sleep(0.5)

def load_rgb(path):
    img = Image.open(path)
    if img.mode != 'RGB':
        img = img.convert('RGB')
    return img

# --- the 9 properties (used for train only) ---

def prop_flip(img):
    if random.random() < THRESHOLD:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
    return img

def prop_rotate(img):
    if random.random() < THRESHOLD:
        angle = random.uniform(-15, 15)
        img = img.rotate(angle, resample=Image.BICUBIC, expand=False, fillcolor=(0, 0, 0))
    return img

def prop_shear(img):
    if random.random() < THRESHOLD:
        w, h = img.size
        shear_deg = random.uniform(-10, 10)
        shear_factor = math.tan(math.radians(shear_deg))
        c = -shear_factor * h / 2
        img = img.transform(img.size, Image.AFFINE, (1, shear_factor, c, 0, 1, 0),
                             resample=Image.BICUBIC, fillcolor=(0, 0, 0))
    return img

def prop_zoom(img):
    # scale < 1.0 = zoom IN (crop tighter, then upscale back to original size)
    # scale > 1.0 = zoom OUT (shrink the whole image, pad the rest with black)
    if random.random() < THRESHOLD:
        w, h = img.size
        scale = random.uniform(0.9, 1.1)
        if scale <= 1.0:
            new_w, new_h = max(1, int(w * scale)), max(1, int(h * scale))
            left = random.randint(0, w - new_w)
            top = random.randint(0, h - new_h)
            img = img.crop((left, top, left + new_w, top + new_h)).resize((w, h), Image.BICUBIC)
        else:
            new_w, new_h = max(1, int(w / scale)), max(1, int(h / scale))
            shrunk = img.resize((new_w, new_h), Image.BICUBIC)
            canvas = Image.new('RGB', (w, h), (0, 0, 0))
            left = random.randint(0, w - new_w)
            top = random.randint(0, h - new_h)
            canvas.paste(shrunk, (left, top))
            img = canvas
    return img

def prop_translate(img):
    if random.random() < THRESHOLD:
        w, h = img.size
        max_dx, max_dy = int(w * 0.1), int(h * 0.1)
        dx = random.randint(-max_dx, max_dx)
        dy = random.randint(-max_dy, max_dy)
        img = img.transform(img.size, Image.AFFINE, (1, 0, dx, 0, 1, dy), fillcolor=(0, 0, 0))
    return img

def prop_brightness(img):
    if random.random() < THRESHOLD:
        factor = random.uniform(0.7, 1.3)
        img = ImageEnhance.Brightness(img).enhance(factor)
    return img

def prop_contrast(img):
    if random.random() < THRESHOLD:
        factor = random.uniform(0.7, 1.3)
        img = ImageEnhance.Contrast(img).enhance(factor)
    return img

def prop_blur(img):
    if random.random() < THRESHOLD:
        radius = random.uniform(0.5, 1.2)
        img = img.filter(ImageFilter.GaussianBlur(radius=radius))
    return img

def prop_saturation(img):
    if random.random() < THRESHOLD:
        factor = random.uniform(0.85, 1.15)
        img = ImageEnhance.Color(img).enhance(factor)
    return img

def augment_once(img):
    img = prop_flip(img)
    img = prop_rotate(img)
    img = prop_shear(img)
    img = prop_zoom(img)
    img = prop_translate(img)
    img = prop_brightness(img)
    img = prop_contrast(img)
    img = prop_blur(img)
    img = prop_saturation(img)
    return img

for split in SRC_SPLITS:
    class_dirs = sorted(d for d in os.listdir(os.path.join(SRC_BASE_DIR, split)) if os.path.isdir(os.path.join(SRC_BASE_DIR, split, d)))
    print(f"=== {split}/ ({len(class_dirs)} classes) ===")

    for class_name in class_dirs:
        src_files = sorted(glob.glob(os.path.join(SRC_BASE_DIR, split, class_name, '*')))
        out_dir = os.path.join(OUT_DIR, split, class_name)
        os.makedirs(out_dir, exist_ok=True)
        clear_dir(out_dir)

        total_saved = 0
        for img_idx, f in enumerate(src_files):
            img = load_rgb(f)
            img.save(os.path.join(out_dir, f"image{img_idx}_original.png"))
            total_saved += 1
            if split == 'train':  # ONLY train gets augmented copies
                for copy_idx in range(1, N_AUG_PER_IMAGE + 1):
                    aug_img = augment_once(img.copy())
                    aug_img.save(os.path.join(out_dir, f"image{img_idx}_copy{copy_idx}.png"))
                    total_saved += 1

        print(f"  {class_name:20s} {len(src_files)} real -> {total_saved} total")

    print()

print("Done. augmented/train (augmented 10x), augmented/val and augmented/test (real, untouched) created.")

=== train/ (15 classes) ===
  Arya_Stark           22 real -> 220 total
  Brandon_Stark        16 real -> 160 total
  Catelyn_Stark        17 real -> 170 total
  Cersei_Lannister     12 real -> 120 total
  Eddard_Stark         26 real -> 260 total
  Jaime_Lannister      15 real -> 150 total
  Joffrey_Baratheon    16 real -> 160 total
  Jon_Snow             17 real -> 170 total
  Maester_Luwin        14 real -> 140 total
  Rickon_Stark         12 real -> 120 total
  Robb_Stark           13 real -> 130 total
  Robert_Baratheon     16 real -> 160 total
  Rodrik_Cassel        14 real -> 140 total
  Sansa_Stark          15 real -> 150 total
  Theon_Greyjoy        15 real -> 150 total

=== val/ (15 classes) ===
  Arya_Stark           7 real -> 7 total
  Brandon_Stark        5 real -> 5 total
  Catelyn_Stark        6 real -> 6 total
  Cersei_Lannister     4 real -> 4 total
  Eddard_Stark         9 real -> 9 total
  Jaime_Lannister      5 real -> 5 total
  Joffrey_Baratheon    5 real -> 5 tota

# Step 3 — Extract FaceNet512 embeddings (frozen, no training)

Runs every image in `data/augmented/train/`, `val/`, `test/` through DeepFace's pretrained **FaceNet512** model, converting each face crop into a fixed 512-number embedding vector. This is feature EXTRACTION only — the FaceNet512 backbone stays completely frozen, nothing about it is trained or modified (per our earlier research: fine-tuning it on this little data would overfit and isn't needed since our faces are the same domain FaceNet was already trained on).

`detector_backend="skip"` is used because our images are already tightly-cropped single faces (from our own MTCNN step) — this tells DeepFace to treat the whole image as the face directly, instead of running its own face detector on it again.

Saves the results to `../models/embeddings/` as `.npy` files (`train_X.npy`/`train_y.npy`, same for val/test) plus `class_names.json` (the ordered list of 15 class names, so numeric labels can be mapped back to character names later). This is the slow step (one forward pass per image, ~2600 images total) — saving to disk means it only ever needs to run once.

`data/augmented/` is only read here, never modified.

In [10]:
import os
import sys
# deepface's own logger prints emoji; Windows' default console encoding (cp1252)
# can't handle them and crashes. reconfigure() actually takes effect on an
# already-running process (unlike setting the PYTHONIOENCODING env var after
# startup, which does not retroactively change an already-open stdout).
try:
    sys.stdout.reconfigure(encoding='utf-8')
    sys.stderr.reconfigure(encoding='utf-8')
except AttributeError:
    pass  # Jupyter's ipykernel OutStream doesn't support reconfigure() and doesn't need it -
    # it already sends output as UTF-8 to the frontend, never through the raw Windows console.
import glob
import time
import json as json_lib
import numpy as np
from deepface import DeepFace

SRC_DIR = '../data/augmented'
SPLITS = ['train', 'val', 'test']
OUT_DIR = '../models/embeddings'
MODEL_NAME = 'Facenet512'

os.makedirs(OUT_DIR, exist_ok=True)

class_names = sorted(d for d in os.listdir(os.path.join(SRC_DIR, 'train'))
                      if os.path.isdir(os.path.join(SRC_DIR, 'train', d)))
print(f"Found {len(class_names)} classes.\n")

with open(os.path.join(OUT_DIR, 'class_names.json'), 'w') as f:
    json_lib.dump(class_names, f, indent=2)

def format_duration(seconds):
    seconds = max(0, int(seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}" if h else f"{m:02d}:{s:02d}"

# Count the total number of images across ALL splits/classes upfront, so we
# can show "X / Y done" and an ETA from the very first image, not just per-split.
total_images = 0
for split in SPLITS:
    for class_name in class_names:
        total_images += len(glob.glob(os.path.join(SRC_DIR, split, class_name, '*')))
print(f"Total images to embed across all splits: {total_images}\n")

processed_count = 0
start_time = time.time()

for split in SPLITS:
    split_dir = os.path.join(OUT_DIR, split)
    os.makedirs(split_dir, exist_ok=True)
    print(f"=== {split}/ ===")

    for class_idx, class_name in enumerate(class_names):
        class_file = os.path.join(split_dir, f'{class_name}.npy')
        files = sorted(glob.glob(os.path.join(SRC_DIR, split, class_name, '*')))

        if os.path.exists(class_file):
            processed_count += len(files)  # already-done images still count toward overall progress
            pct = 100 * processed_count / total_images if total_images else 0
            print(f"  {class_name:20s} already done, skipping (resume)  [{processed_count}/{total_images}, {pct:.1f}%]")
            continue

        class_embeddings = []
        for f in files:
            result = DeepFace.represent(
                img_path=f,
                model_name=MODEL_NAME,
                detector_backend='skip',  # already a clean, single face crop - don't re-detect
                enforce_detection=False,
            )
            class_embeddings.append(result[0]['embedding'])
            processed_count += 1

            elapsed = time.time() - start_time
            rate = processed_count / elapsed if elapsed > 0 else 0
            remaining_images = total_images - processed_count
            eta = remaining_images / rate if rate > 0 else 0
            pct = 100 * processed_count / total_images if total_images else 0

            sys.stdout.write(
                f"\r    [{processed_count}/{total_images}, {pct:5.1f}%] "
                f"{rate:.2f} img/s | elapsed {format_duration(elapsed)} | ETA {format_duration(eta)}   "
            )
            sys.stdout.flush()

        class_embeddings = np.array(class_embeddings, dtype=np.float32)
        np.save(class_file, class_embeddings)  # saved immediately - a crash later only loses the CURRENT class, not the whole split
        print(f"\r  {class_name:20s} {len(files)} images embedded -> saved" + " " * 40)

    # combine this split's per-class files into the final train_X.npy/train_y.npy format
    X, y = [], []
    for class_idx, class_name in enumerate(class_names):
        class_embeddings = np.load(os.path.join(split_dir, f'{class_name}.npy'))
        X.append(class_embeddings)
        y.append(np.full(len(class_embeddings), class_idx, dtype=np.int64))

    X = np.concatenate(X, axis=0)
    y = np.concatenate(y, axis=0)
    np.save(os.path.join(OUT_DIR, f'{split}_X.npy'), X)
    np.save(os.path.join(OUT_DIR, f'{split}_y.npy'), y)
    print(f"  -> combined into {split}_X.npy / {split}_y.npy, shape {X.shape}\n")

total_elapsed = time.time() - start_time
print(f"Done in {format_duration(total_elapsed)}. Embeddings saved to {OUT_DIR}/")


Found 15 classes.

Total images to embed across all splits: 2561

=== train/ ===
  Arya_Stark           220 images embedded -> saved                                        
  Brandon_Stark        160 images embedded -> saved                                        
  Catelyn_Stark        170 images embedded -> saved                                        
  Cersei_Lannister     120 images embedded -> saved                                        
  Eddard_Stark         260 images embedded -> saved                                        
  Jaime_Lannister      150 images embedded -> saved                                        
  Joffrey_Baratheon    160 images embedded -> saved                                        
  Jon_Snow             170 images embedded -> saved                                        
  Maester_Luwin        140 images embedded -> saved                                        
  Rickon_Stark         120 images embedded -> saved                                       

# Step 4 — Nearest-centroid baseline (no training, zero trainable parameters)

For each of the 15 classes, computes the CENTROID — the average of that class's embedding vectors in the train set — giving exactly 15 vectors, one per character. Then for every val AND test example, computes cosine similarity against all 15 centroids and predicts whichever one is highest.

No training happens here at all — this is the simpler alternative to a trained dense classifier we discussed, testing whether the FaceNet512 embeddings already separate our characters well enough on their own, before we invest in training anything.

Reports val and test results separately — test is the untouched, never-before-looked-at set, so its number is the more honest one, but comparing both shows whether performance holds up consistently or val was a lucky/unlucky split.

In [11]:
import os
import numpy as np
import json

with open('../models/embeddings/class_names.json') as f:
    class_names = json.load(f)

train_X = np.load('../models/embeddings/train_X.npy')
train_y = np.load('../models/embeddings/train_y.npy')

n_classes = len(class_names)

CENTROID_DIR = '../models/centroids'
os.makedirs(CENTROID_DIR, exist_ok=True)

# One centroid per class: the mean of that class's train embeddings.
# Reuses saved vectors if already computed - only recomputes+saves if missing.
centroids = np.zeros((n_classes, train_X.shape[1]), dtype=np.float32)
all_saved = all(os.path.exists(os.path.join(CENTROID_DIR, f'{name}.npy')) for name in class_names)

if all_saved:
    print(f"Loading existing centroids from {CENTROID_DIR}/")
    for c, name in enumerate(class_names):
        centroids[c] = np.load(os.path.join(CENTROID_DIR, f'{name}.npy'))
else:
    print(f"Computing centroids and saving to {CENTROID_DIR}/")
    for c, name in enumerate(class_names):
        centroids[c] = train_X[train_y == c].mean(axis=0)
        np.save(os.path.join(CENTROID_DIR, f'{name}.npy'), centroids[c])
print()

def cosine_sim_matrix(A, B):
    A_norm = A / np.linalg.norm(A, axis=1, keepdims=True)
    B_norm = B / np.linalg.norm(B, axis=1, keepdims=True)
    return A_norm @ B_norm.T

def evaluate(split_name):
    X = np.load(f'../models/embeddings/{split_name}_X.npy')
    y = np.load(f'../models/embeddings/{split_name}_y.npy')

    sims = cosine_sim_matrix(X, centroids)  # shape (n_examples, n_classes)
    predictions = np.argmax(sims, axis=1)

    correct = (predictions == y)
    n_correct = int(correct.sum())
    n_total = len(y)
    n_wrong = n_total - n_correct

    print(f"=== {split_name} ===")
    print(f"Correct: {n_correct}/{n_total} ({100*n_correct/n_total:.1f}%)")
    print(f"Wrong:   {n_wrong}/{n_total} ({100*n_wrong/n_total:.1f}%)")
    print()

    print("Per-class breakdown:")
    for c, name in enumerate(class_names):
        mask = y == c
        n_class_total = int(mask.sum())
        n_class_correct = int(correct[mask].sum())
        flag = '  <-- errors here' if n_class_correct < n_class_total else ''
        print(f"  {name:20s} {n_class_correct}/{n_class_total} correct{flag}")

    print()
    print("Misclassified examples:")
    wrong_idx = np.where(~correct)[0]
    if len(wrong_idx) == 0:
        print("  (none)")
    for idx in wrong_idx:
        true_name = class_names[y[idx]]
        pred_name = class_names[predictions[idx]]
        print(f"  true={true_name:20s} predicted={pred_name:20s} similarity={sims[idx, predictions[idx]]:.4f}")
    print()

    return n_correct, n_total

val_correct, val_total = evaluate('val')
test_correct, test_total = evaluate('test')

print("=== summary ===")
print(f"val:  {val_correct}/{val_total} ({100*val_correct/val_total:.1f}%)")
print(f"test: {test_correct}/{test_total} ({100*test_correct/test_total:.1f}%)")

Computing centroids and saving to ../models/centroids/

=== val ===
Correct: 80/80 (100.0%)
Wrong:   0/80 (0.0%)

Per-class breakdown:
  Arya_Stark           7/7 correct
  Brandon_Stark        5/5 correct
  Catelyn_Stark        6/6 correct
  Cersei_Lannister     4/4 correct
  Eddard_Stark         9/9 correct
  Jaime_Lannister      5/5 correct
  Joffrey_Baratheon    5/5 correct
  Jon_Snow             6/6 correct
  Maester_Luwin        5/5 correct
  Rickon_Stark         4/4 correct
  Robb_Stark           4/4 correct
  Robert_Baratheon     5/5 correct
  Rodrik_Cassel        5/5 correct
  Sansa_Stark          5/5 correct
  Theon_Greyjoy        5/5 correct

Misclassified examples:
  (none)

=== test ===
Correct: 80/81 (98.8%)
Wrong:   1/81 (1.2%)

Per-class breakdown:
  Arya_Stark           6/7 correct  <-- errors here
  Brandon_Stark        5/5 correct
  Catelyn_Stark        6/6 correct
  Cersei_Lannister     4/4 correct
  Eddard_Stark         8/8 correct
  Jaime_Lannister      5/5 correct

# Step 6 — Train a small dense classifier on top of the FaceNet512 embeddings

FaceNet512 stays completely frozen — nothing here touches it. This trains only a small new head on top of the already-extracted embeddings:

```
Input(512) -> L2-Normalize -> Dense(64, relu) -> Dropout(0.5) -> Dense(15, softmax)
```

**L2-normalize first (via `keras.layers.UnitNormalization`, a real built-in layer)**: cosine similarity (what the Step 4 centroid baseline used) implicitly normalizes vectors before comparing them. Without this, the classifier would waste some capacity learning to handle varying vector magnitudes instead of focusing purely on direction, where the identity information actually lives. (Originally built with a raw `Lambda` layer doing this - switched to `UnitNormalization` after discovering Keras 3 refuses to safely deserialize saved models containing arbitrary Python lambdas, and even with that override, can't infer the Lambda's output shape on reload. UnitNormalization is a registered layer type, so it has neither problem.)

**64 units, not more**: given the embeddings are already well-separated (verified in the cosine-similarity checks earlier), the task is closer to "draw a boundary between already-clustered points" than "learn to distinguish faces from scratch" — more capacity mainly risks overfitting on our ~162 real underlying images.

**`class_weight`**: computed from the real train label distribution (`Eddard_Stark` at 0.623 up to `Cersei_Lannister` at 1.350) — so mistakes on underrepresented classes count proportionally more during training.

**Stopping**: `EarlyStopping` monitors `val_loss` (more sensitive than `val_accuracy` with only 81 val examples), patience 15, `restore_best_weights=True` so the final model is whichever epoch had the best val_loss, not just wherever training happened to stop. `ModelCheckpoint` also persists that best epoch to `facenet_classifier.keras` on disk as a belt-and-suspenders backup.

Uses the standalone `keras` package (not `tensorflow.keras`) throughout, consistent with the fix from Step 5, since `deepface`'s use of `tf_keras` earlier in this session can interfere with `tensorflow.keras`'s model handling.

In [3]:
import json
import numpy as np
import keras
from keras import layers

with open('../models/embeddings/class_names.json') as f:
    class_names = json.load(f)
n_classes = len(class_names)

train_X = np.load('../models/embeddings/train_X.npy')
train_y = np.load('../models/embeddings/train_y.npy')
val_X = np.load('../models/embeddings/val_X.npy')
val_y = np.load('../models/embeddings/val_y.npy')
test_X = np.load('../models/embeddings/test_X.npy')
test_y = np.load('../models/embeddings/test_y.npy')

# --- class weights, computed from the real train label distribution ---
n_train_total = len(train_y)
class_weight = {}
for c in range(n_classes):
    count = int((train_y == c).sum())
    class_weight[c] = n_train_total / (n_classes * count)
print("Class weights:")
for c, name in enumerate(class_names):
    print(f"  {name:20s} {class_weight[c]:.3f}")
print()

# --- model: Input(512) -> L2-Normalize -> Dense(64, relu) -> Dropout(0.5) -> Dense(15, softmax) ---
inputs = keras.Input(shape=(512,), name='facenet_embedding')
x = layers.UnitNormalization(axis=1, name='l2_normalize')(inputs)  # built-in layer - no Lambda deserialization issues
x = layers.Dense(
    64, activation='relu', name='hidden',
    kernel_regularizer=keras.regularizers.l2(1e-4),  # keeps weight/logit magnitude from growing unbounded
)(x)
x = layers.Dropout(0.5, name='dropout')(x)
outputs = layers.Dense(n_classes, activation='softmax', name='predictions')(x)
model = keras.Model(inputs, outputs, name='facenet_dense_classifier')

# SparseCategoricalCrossentropy does NOT support label_smoothing (only the
# one-hot CategoricalCrossentropy does) - so labels are one-hot encoded just
# for this loss; evaluation later still uses the original sparse train_y/
# val_y/test_y (argmax comparison works the same either way).
train_y_onehot = keras.utils.to_categorical(train_y, num_classes=n_classes)
val_y_onehot = keras.utils.to_categorical(val_y, num_classes=n_classes)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),  # softens targets - no class ever has to hit exactly 1.0
    metrics=['accuracy'],
)
model.summary()

# --- callbacks: stop on val_loss plateau, keep the BEST epoch's weights ---
# patience lowered from 15 to 7 - stop as soon as val_loss genuinely stops
# improving instead of grinding on for the full 100 epochs regardless
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint('../models/facenet_classifier.keras', monitor='val_loss', save_best_only=True),
]

history = model.fit(
    train_X, train_y_onehot,
    validation_data=(val_X, val_y_onehot),
    epochs=20,
    batch_size=32,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=2,
)

n_epochs_run = len(history.history['loss'])
best_val_loss = min(history.history['val_loss'])
print(f"\nTraining stopped after {n_epochs_run} epochs. Best val_loss: {best_val_loss:.4f}\n")


# --- evaluate, same report format as Steps 4 and 5 for a direct comparison ---
def evaluate_classifier(X, y, split_name):
    probs = model.predict(X, verbose=0)
    predictions = np.argmax(probs, axis=1)
    confidences = np.max(probs, axis=1)

    correct = (predictions == y)
    n_correct = int(correct.sum())
    n_total = len(y)
    n_wrong = n_total - n_correct

    print(f"=== {split_name} (trained dense classifier) ===")
    print(f"Correct: {n_correct}/{n_total} ({100*n_correct/n_total:.1f}%)")
    print(f"Wrong:   {n_wrong}/{n_total} ({100*n_wrong/n_total:.1f}%)")
    print()

    print("Per-class breakdown:")
    for c, name in enumerate(class_names):
        mask = y == c
        n_class_total = int(mask.sum())
        n_class_correct = int(correct[mask].sum())
        flag = '  <-- errors here' if n_class_correct < n_class_total else ''
        print(f"  {name:20s} {n_class_correct}/{n_class_total} correct{flag}")

    print()
    print("Misclassified examples:")
    wrong_idx = np.where(~correct)[0]
    if len(wrong_idx) == 0:
        print("  (none)")
    for idx in wrong_idx:
        true_name = class_names[y[idx]]
        pred_name = class_names[predictions[idx]]
        print(f"  true={true_name:20s} predicted={pred_name:20s} confidence={confidences[idx]:.4f}")
    print()

    return n_correct, n_total

val_correct, val_total = evaluate_classifier(val_X, val_y, 'val')
test_correct, test_total = evaluate_classifier(test_X, test_y, 'test')

print("=== summary (trained dense classifier) ===")
print(f"val:  {val_correct}/{val_total} ({100*val_correct/val_total:.1f}%)")
print(f"test: {test_correct}/{test_total} ({100*test_correct/test_total:.1f}%)")

Class weights:
  Arya_Stark           0.727
  Brandon_Stark        1.000
  Catelyn_Stark        0.941
  Cersei_Lannister     1.333
  Eddard_Stark         0.615
  Jaime_Lannister      1.067
  Joffrey_Baratheon    1.000
  Jon_Snow             0.941
  Maester_Luwin        1.143
  Rickon_Stark         1.333
  Robb_Stark           1.231
  Robert_Baratheon     1.000
  Rodrik_Cassel        1.143
  Sansa_Stark          1.067
  Theon_Greyjoy        1.067



Model: "facenet_dense_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ facenet_embedding (InputLayer)  │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ l2_normalize                    │ (None, 512)            │             0 │
│ (UnitNormalization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden (Dense)                  │ (None, 64)             │        32,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 15)             │           975 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 33,807 (132.06 KB)

 Trainable params: 33,807 (132.06 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
75/75 - 2s - 27ms/step - accuracy: 0.7000 - loss: 2.2291 - val_accuracy: 0.9750 - val_loss: 1.5713
Epoch 2/20
75/75 - 0s - 5ms/step - accuracy: 0.9179 - loss: 1.2922 - val_accuracy: 1.0000 - val_loss: 0.8422
Epoch 3/20
75/75 - 0s - 4ms/step - accuracy: 0.9496 - loss: 0.9504 - val_accuracy: 1.0000 - val_loss: 0.6921
Epoch 4/20
75/75 - 0s - 5ms/step - accuracy: 0.9588 - loss: 0.8732 - val_accuracy: 1.0000 - val_loss: 0.6638
Epoch 5/20
75/75 - 0s - 5ms/step - accuracy: 0.9658 - loss: 0.8403 - val_accuracy: 1.0000 - val_loss: 0.6531
Epoch 6/20
75/75 - 0s - 6ms/step - accuracy: 0.9742 - loss: 0.8170 - val_accuracy: 1.0000 - val_loss: 0.6469
Epoch 7/20
75/75 - 0s - 5ms/step - accuracy: 0.9729 - loss: 0.8045 - val_accuracy: 1.0000 - val_loss: 0.6429
Epoch 8/20
75/75 - 0s - 4ms/step - accuracy: 0.9796 - loss: 0.7992 - val_accuracy: 1.0000 - val_loss: 0.6411
Epoch 9/20
75/75 - 0s - 4ms/step - accuracy: 0.9825 - loss: 0.7826 - val_accuracy: 1.0000 - val_loss: 0.6400
Epoch 10/20
75/75 

# Step 7 — How often do centroid and classifier disagree WITH EACH OTHER?

Different question from Steps 4/6: those compared each method against the TRUE label. This compares the two methods' predictions **against each other**, regardless of which (if either) is actually correct — this is exactly the number that determines how often `facenet_app.py`'s agreement gate falls back to "not clear enough", since that gate triggers on disagreement alone.

Loads the saved `../models/centroids/` and `facenet_classifier.keras` — both untouched, nothing trained or modified here.

In [1]:
import os
import json
import numpy as np
import keras

with open('../models/embeddings/class_names.json') as f:
    class_names = json.load(f)
n_classes = len(class_names)

centroids = np.zeros((n_classes, 512), dtype=np.float32)
for c, name in enumerate(class_names):
    centroids[c] = np.load(os.path.join('../models/centroids', f'{name}.npy'))

classifier = keras.models.load_model('../models/facenet_classifier.keras')

def cosine_sim_matrix(A, B):
    A_norm = A / np.linalg.norm(A, axis=1, keepdims=True)
    B_norm = B / np.linalg.norm(B, axis=1, keepdims=True)
    return A_norm @ B_norm.T

def compare(split_name):
    X = np.load(f'../models/embeddings/{split_name}_X.npy')
    y = np.load(f'../models/embeddings/{split_name}_y.npy')

    sims = cosine_sim_matrix(X, centroids)
    centroid_preds = np.argmax(sims, axis=1)

    probs = classifier.predict(X, verbose=0)
    classifier_preds = np.argmax(probs, axis=1)
    classifier_confidences = np.max(probs, axis=1)

    agree = (centroid_preds == classifier_preds)
    n_agree = int(agree.sum())
    n_total = len(y)
    n_disagree = n_total - n_agree

    print(f"=== {split_name} ===")
    print(f"Agree:    {n_agree}/{n_total} ({100*n_agree/n_total:.1f}%)")
    print(f"Disagree: {n_disagree}/{n_total} ({100*n_disagree/n_total:.1f}%)")
    print()

    if n_disagree > 0:
        print("Disagreement cases:")
        for idx in np.where(~agree)[0]:
            true_name = class_names[y[idx]]
            centroid_name = class_names[centroid_preds[idx]]
            classifier_name = class_names[classifier_preds[idx]]
            print(f"  true={true_name:20s} centroid_says={centroid_name:20s} "
                  f"classifier_says={classifier_name:20s} classifier_conf={classifier_confidences[idx]:.3f}")
    print()

    return n_agree, n_disagree, n_total

val_agree, val_disagree, val_total = compare('val')
test_agree, test_disagree, test_total = compare('test')

print("=== summary ===")
print(f"val:  {val_disagree}/{val_total} disagreements ({100*val_disagree/val_total:.1f}%)")
print(f"test: {test_disagree}/{test_total} disagreements ({100*test_disagree/test_total:.1f}%)")


def print_full_table(split_name):
    X = np.load(f'../models/embeddings/{split_name}_X.npy')
    y = np.load(f'../models/embeddings/{split_name}_y.npy')

    sims = cosine_sim_matrix(X, centroids)
    centroid_preds = np.argmax(sims, axis=1)
    centroid_top_sims = np.max(sims, axis=1)

    probs = classifier.predict(X, verbose=0)
    classifier_preds = np.argmax(probs, axis=1)
    classifier_confidences = np.max(probs, axis=1)

    print(f"=== {split_name}: all {len(y)} examples ===")
    header = (f"{'idx':>3s}  {'true':20s}  {'centroid_pred':20s}  {'centroid_sim':>12s}  "
              f"{'classifier_pred':20s}  {'classifier_conf':>15s}  {'match':>5s}")
    print(header)
    for idx in range(len(y)):
        true_name = class_names[y[idx]]
        centroid_name = class_names[centroid_preds[idx]]
        classifier_name = class_names[classifier_preds[idx]]
        match = 'YES' if centroid_name == classifier_name else 'NO'
        print(f"{idx:3d}  {true_name:20s}  {centroid_name:20s}  {centroid_top_sims[idx]:12.4f}  "
              f"{classifier_name:20s}  {classifier_confidences[idx]:15.4f}  {match:>5s}")
    print()

print_full_table('val')

c:\Users\itsme\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.0) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


=== val ===
Agree:    80/80 (100.0%)
Disagree: 0/80 (0.0%)


=== test ===
Agree:    80/81 (98.8%)
Disagree: 1/81 (1.2%)

Disagreement cases:
  true=Arya_Stark           centroid_says=Rickon_Stark         classifier_says=Jon_Snow             classifier_conf=0.325

=== summary ===
val:  0/80 disagreements (0.0%)
test: 1/81 disagreements (1.2%)
=== val: all 80 examples ===
idx  true                  centroid_pred         centroid_sim  classifier_pred       classifier_conf  match
  0  Arya_Stark            Arya_Stark                  0.8386  Arya_Stark                     0.9123    YES
  1  Arya_Stark            Arya_Stark                  0.8175  Arya_Stark                     0.9545    YES
  2  Arya_Stark            Arya_Stark                  0.7108  Arya_Stark                     0.9158    YES
  3  Arya_Stark            Arya_Stark                  0.8696  Arya_Stark                     0.9426    YES
  4  Arya_Stark            Arya_Stark                  0.6718  Arya_Stark              